# **Причинный лес и честное расщепление**

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school), урок «Причинный лес и честное расщепление».

В уроке честное расщепление (honest splitting) было рассуждением: если одни и те же объекты выбирают расщепление и оценивают эффект в листе, оценка эффекта смещается в сторону преувеличения, а если роли разделить, смещение исчезает и появляются доверительные интервалы. Здесь мы проверим это числами:

1. одно дерево на данных, где эффекта нет ни у кого: что находит адаптивное дерево и что остается после честной переоценки;
2. причинный лес с честным расщеплением и без него: точность, покрытие доверительных интервалов, роль минимального размера листа;
3. лес как способ подобрать соседей: веса $\alpha_i(x)$ из generalized random forests;
4. где лес ломается: нарушенная позитивность;
5. применение к данным о школьной программе growth mindset.

Тетрадь идет на процессоре; весь прогон занимает около минуты.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from econml.grf import CausalForest
from econml.dml import CausalForestDML

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # ложные предупреждения numpy на некоторых процессорах
SEED = 0
pd.set_option('display.width', 140)

## 1. Одно дерево там, где эффекта нет

Рандомизированный эксперимент: 2000 объектов, десять признаков $X\sim\mathcal N(0,I)$, воздействие получает случайная половина. Исход зависит только от первого признака, $Y=x_0+\varepsilon$, а эффекта воздействия нет ни у кого: $\tau(x)=0$. Любая подгруппа с большим эффектом, которую мы найдем, найдена в шуме.

Причинное дерево соберем из обычного регрессионного дерева. При вероятности воздействия $p=0{,}5$ у преобразованного исхода $Y^*=Y\,\frac{T-p}{p(1-p)}$ условное среднее равно эффекту: $\mathbb E[Y^*\mid X=x]=\tau(x)$. Дерево, обученное предсказывать $Y^*$, режет выборку там, где эффект, по его мнению, различается. Эффект в листе считаем как разность средних исходов воздействованных и контрольных объектов.

Половина A выбирает структуру дерева (восемь листьев, в каждом не меньше 50 объектов), половина B для честной переоценки пока не используется.

In [ ]:
def null_experiment(seed, n=2000, d=10):
    r = np.random.default_rng(seed)
    Xn = r.normal(size=(n, d))
    Tn = r.binomial(1, 0.5, n)
    Yn = Xn[:, 0] + r.normal(size=n)          # эффекта нет: tau(x) = 0
    return Xn, Tn, Yn


def leaf_effects(leaves, t, y):
    # разность средних исходов воздействованных и контрольных в каждом листе
    ids = np.unique(leaves)
    return ids, np.array([y[(leaves == l) & (t == 1)].mean() - y[(leaves == l) & (t == 0)].mean() for l in ids])


def causal_tree(Xn, Tn, Yn, A):
    y_star = Yn * (Tn - 0.5) / 0.25            # преобразованный исход: E[Y* | x] = tau(x)
    return DecisionTreeRegressor(max_leaf_nodes=8, min_samples_leaf=50, random_state=SEED).fit(Xn[A], y_star[A])


Xn, Tn, Yn = null_experiment(SEED)
A = np.arange(len(Xn)) < len(Xn) // 2          # половина A: выбор расщеплений
B = ~A                                         # половина B: честная переоценка
tree = causal_tree(Xn, Tn, Yn, A)
ids, adaptive = leaf_effects(tree.apply(Xn[A]), Tn[A], Yn[A])
print('эффект в листьях, посчитанный на той же половине A:', np.round(adaptive, 2))

**Задание 1. Переоцените эффект в тех же листьях честно: разложите по листьям дерева `tree` объекты половины B и посчитайте в каждом листе разность средних исходов функцией `leaf_effects` (массив `honest`). Затем повторите весь эксперимент для сидов 100–199: для каждого сида функцией `null_experiment` сгенерируйте данные, обучите дерево на половине A, найдите лист с наибольшим эффектом на половине A и запишите его эффект на половине A и на половине B. Выведите средние двух величин по 100 сидам.**

In [ ]:
# Ваш код здесь

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
bins = np.linspace(-1.5, 2.5, 41)
ax.hist(best_a, bins=bins, alpha=0.6, color='#c0392b', label='estimated on half A (adaptive)')
ax.hist(best_b, bins=bins, alpha=0.6, color='#3b6ea5', label='re-estimated on half B (honest)')
ax.axvline(0, color='black', lw=1, ls=':', label='true effect = 0')
ax.set_xlabel('Effect in the leaf with the largest adaptive estimate')
ax.set_ylabel('Seeds (of 100)')
ax.set_title('No effect anywhere: the "best subgroup" of one causal tree')
ax.legend(fontsize=8)
plt.show()

## 2. Лес: честное расщепление, покрытие интервалов и размер листа

Теперь эффект есть и различается по объектам. Возьмем постановку из статьи [Wager и Athey (2018)](https://arxiv.org/abs/1510.04342): пять признаков $X\sim U[0,1]^5$, воздействие у случайной половины, эффект

$$\tau(x)=\varsigma(x_0)\,\varsigma(x_1),\qquad \varsigma(u)=1+\frac{1}{1+e^{-20(u-1/3)}},$$

то есть от 1 до 4 в зависимости от двух первых признаков, исход $Y=2x_2+T\tau(X)+\varepsilon$. Обучаем `econml.grf.CausalForest` на 4000 объектах и проверяем на 2000 других, где истинный эффект известен: RMSE оценки эффекта, доля объектов, у которых истинный эффект попал в 95-процентный доверительный интервал (покрытие), и средняя ширина интервала.

Два гиперпараметра: `honest` (честное расщепление внутри каждого дерева) и `min_samples_leaf` (минимальное число объектов в листе).

In [ ]:
def wager_athey(n, r):
    Xw = r.uniform(0, 1, (n, 5))
    Tw = r.binomial(1, 0.5, n)
    s = lambda u: 1 + 1 / (1 + np.exp(-20 * (u - 1 / 3)))
    tw = s(Xw[:, 0]) * s(Xw[:, 1])
    return Xw, Tw, 2 * Xw[:, 2] + Tw * tw + r.normal(size=n), tw


r = np.random.default_rng(1)
Xa, Ta, Ya, tau_a = wager_athey(4000, r)       # обучение
Xb, Tb, Yb, tau_b = wager_athey(2000, r)       # проверка: нужен истинный эффект
print(f'истинный эффект на проверке: от {tau_b.min():.2f} до {tau_b.max():.2f}, среднее {tau_b.mean():.2f}')


def forest_report(honest, min_samples_leaf):
    cf = CausalForest(n_estimators=1000, min_samples_leaf=min_samples_leaf, honest=honest, random_state=SEED).fit(Xa, Ta, Ya)
    pt, lo, hi = (a.ravel() for a in cf.predict(Xb, interval=True, alpha=0.05))
    return {'RMSE ↓': np.sqrt(np.mean((pt - tau_b) ** 2)),
            'покрытие 95%-интервала': np.mean((lo <= tau_b) & (tau_b <= hi)),
            'средняя ширина интервала': np.mean(hi - lo)}

**Задание 2. Посчитайте `forest_report` для `honest` из `(True, False)` и `min_samples_leaf` из `(5, 20, 50)` и соберите таблицу `grid` (строки — пары значений, столбцы — три величины). При каком сочетании покрытие ближе всего к номинальным 95%? Что происходит с покрытием честного леса при `min_samples_leaf=50`?**

In [ ]:
# Ваш код здесь

Покрытие ниже номинального у честного леса с крупными листьями — следствие смещения: в большом листе собираются объекты с разным эффектом, оценка в точке сглаживается, и интервал, построенный вокруг смещенной оценки, промахивается мимо истинного значения. Интервал нечестного леса шире и покрывает истинный эффект чаще номинального, но это не гарантия: асимптотическая теория интервалов в [Wager и Athey (2018)](https://arxiv.org/abs/1510.04342) доказана только для честных деревьев.

## 3. Лес как способ подобрать соседей

Generalized random forests ([Athey, Tibshirani и Wager, 2019](https://arxiv.org/abs/1610.01271)) читают лес так: для точки $x$ каждый обучающий объект $i$ получает вес $\alpha_i(x)$ — долю деревьев, где он попал в один лист с $x$, нормированную на размер листа. Эффект в $x$ — решение взвешенного оценивающего уравнения; для бинарного воздействия это взвешенная ковариация, деленная на взвешенную дисперсию воздействия:

$$\hat\tau(x)=\frac{\sum_i\alpha_i(x)(T_i-\bar T_\alpha)(Y_i-\bar Y_\alpha)}{\sum_i\alpha_i(x)(T_i-\bar T_\alpha)^2}.$$

Восстановим веса по листьям леса (`apply`) и подвыборкам деревьев (`get_subsample_inds`). Внутри каждого дерева половина его подвыборки только выбирает расщепления; мы для простоты считаем веса по всей подвыборке, поэтому оценка по весам совпадет с прогнозом леса приближенно.

In [ ]:
cf = CausalForest(n_estimators=1000, min_samples_leaf=20, honest=True, random_state=SEED).fit(Xa, Ta, Ya)


def grf_weights(forest, Xtrain, x0):
    leaves, leaf0 = forest.apply(Xtrain), forest.apply(x0[None])[0]
    alpha = np.zeros(len(Xtrain))
    for b, sub in enumerate(forest.get_subsample_inds()):
        in_sub = np.zeros(len(Xtrain), bool)
        in_sub[sub] = True
        same = in_sub & (leaves[:, b] == leaf0[b])     # объекты подвыборки дерева b в листе точки x0
        alpha[same] += 1 / same.sum()
    return alpha / len(forest.get_subsample_inds())


def weighted_effect(alpha, t, y):
    tb, yb = (alpha * t).sum(), (alpha * y).sum()
    return (alpha * (t - tb) * (y - yb)).sum() / (alpha * (t - tb) ** 2).sum()


points = np.array([[0.15, 0.15, 0.5, 0.5, 0.5], [0.8, 0.8, 0.5, 0.5, 0.5]])
s = lambda u: 1 + 1 / (1 + np.exp(-20 * (u - 1 / 3)))
alphas = [grf_weights(cf, Xa, p) for p in points]
for p, a in zip(points, alphas):
    print(f'x0={p[0]}, x1={p[1]}: истинный эффект {s(p[0]) * s(p[1]):.2f}, прогноз леса {cf.predict(p[None]).item():.2f}, '
          f'оценка по весам {weighted_effect(a, Ta, Ya):.2f}, объектов с весом > 0: {(a > 0).sum()}')

Где лежат соседи. Слева — веса обучающих объектов в плоскости двух признаков, от которых эффект зависит ($x_0$, $x_1$); справа — в плоскости признака исхода $x_2$ и признака-шума $x_3$. Размер и цвет точки — вес $\alpha_i(x)$, крестик — сама точка $x$.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8.5))
for row, (p, a) in enumerate(zip(points, alphas)):
    for ax, (i, j) in zip(axes[row], ((0, 1), (2, 3))):
        m = a > 0
        ax.scatter(Xa[~m, i], Xa[~m, j], s=2, color='#d0d0d0')
        ax.scatter(Xa[m, i], Xa[m, j], s=4 + 4000 * a[m], c=a[m], cmap='viridis', alpha=0.8)
        ax.scatter(p[i], p[j], marker='x', s=120, color='#c0392b', lw=2.5)
        ax.set_xlabel(f'x{i}')
        ax.set_ylabel(f'x{j}')
        ax.set_title(f'Point x0={p[0]}, x1={p[1]}: weights in (x{i}, x{j})')
fig.tight_layout()
plt.show()

**Задание 3. Для каждой из двух точек посчитайте, какая доля суммарного веса $\sum_i\alpha_i(x)$ приходится на обучающие объекты, у которых оба признака эффекта $x_0$ и $x_1$ отличаются от координат точки не больше чем на 0,2. Сравните с долей таких объектов во всей обучающей выборке.**

In [ ]:
# Ваш код здесь

## 4. Где лес ломается: нарушенная позитивность

Урок предупреждает: там, где нарушена позитивность, лес экстраполирует и выдает уверенное число. Проверим. Пусть воздействие назначали по правилу: при $x_0<0{,}2$ его получали все, при $x_0>0{,}8$ — никто, в середине — случайная половина. Эффект $\tau(x)=1+3x_0$ растет по тому же признаку, исход $Y=2x_1+T\tau(X)+\varepsilon$. В крайних областях нет ни одного объекта из противоположной группы, и разность исходов там не наблюдается вообще.

Обучаем `CausalForestDML` (модели исхода и склонности — бустинг) на 4000 объектах и сравниваем RMSE, покрытие и ширину интервалов в трех областях на 2000 проверочных объектах.

In [ ]:
r = np.random.default_rng(2)
Xp = r.uniform(0, 1, (4000, 5))
e_true = np.where(Xp[:, 0] < 0.2, 1.0, np.where(Xp[:, 0] > 0.8, 0.0, 0.5))
Tp = r.binomial(1, e_true)
Yp = 2 * Xp[:, 1] + Tp * (1 + 3 * Xp[:, 0]) + r.normal(size=4000)
Xq = r.uniform(0, 1, (2000, 5))
tau_q = 1 + 3 * Xq[:, 0]

cfp = CausalForestDML(model_y=GradientBoostingRegressor(random_state=SEED), model_t=GradientBoostingClassifier(random_state=SEED),
                      discrete_treatment=True, n_estimators=1000, min_samples_leaf=10, random_state=SEED).fit(Yp, Tp, X=Xp)
pt = cfp.effect(Xq)
lo, hi = cfp.effect_interval(Xq, alpha=0.05)
regions = {'x0 < 0.2: только воздействие': Xq[:, 0] < 0.2,
           '0.2 ≤ x0 ≤ 0.8: обе группы': (Xq[:, 0] >= 0.2) & (Xq[:, 0] <= 0.8),
           'x0 > 0.8: только контроль': Xq[:, 0] > 0.8}
pos = pd.DataFrame({k: {'RMSE ↓': np.sqrt(np.mean((pt[m] - tau_q[m]) ** 2)),
                        'покрытие 95%-интервала': np.mean((lo[m] <= tau_q[m]) & (tau_q[m] <= hi[m])),
                        'средняя ширина интервала': np.mean(hi[m] - lo[m])} for k, m in regions.items()}).T
print(pos.round(3))

In [ ]:
order = np.argsort(Xq[:, 0])
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.fill_between(Xq[order, 0], lo[order], hi[order], color='#3b6ea5', alpha=0.25, label='95% interval')
ax.plot(Xq[order, 0], pt[order], color='#3b6ea5', lw=1, label='Causal forest estimate')
ax.plot(Xq[order, 0], tau_q[order], color='#c0392b', lw=2, label='True effect 1 + 3·x0')
for xv in (0.2, 0.8):
    ax.axvline(xv, color='black', ls=':', lw=1)
ax.text(0.02, 4.3, 'treated only', fontsize=9)
ax.text(0.82, 1.2, 'control only', fontsize=9)
ax.set_xlabel('x0')
ax.set_ylabel('Effect')
ax.set_title('Causal forest without overlap: intervals barely widen where no comparison exists')
ax.legend(fontsize=8, loc='upper center')
plt.show()

**Задание 4. По таблице выше: во сколько раз покрытие в области «только воздействие» ниже, чем в области, где есть обе группы? Ошибка оценки (RMSE) в крайних областях выше, чем в середине, в 4–5 раз, а средняя ширина интервала — только в 1,4–1,6 раза. Почему интервал так мало реагирует на то, что данных для сравнения там нет?**

**Ваш ответ:**

## 5. Применение: программа growth mindset в школах

Данные — симуляция по National Study of Learning Mindsets, которую организаторы семинара о методах оценки неоднородности эффекта на конференции ACIC 2018 подготовили для сравнения методов; [Athey и Wager (2019)](https://arxiv.org/abs/1902.07409) разобрали их причинным лесом. 10391 ученица и ученик из 76 школ; воздействие `intervention` — короткая онлайн-программа о том, что способности развиваются; исход `achievement_score` — стандартизованная успеваемость. Воздействие назначено не случайно: оно связано с ожиданиями успеха `success_expect` и другими признаками, поэтому модели исхода и склонности здесь нужны всерьез.

Признаки ученика — `success_expect`, `ethnicity`, `gender`, `frst_in_family`; признаки школы — `school_urbanicity`, `school_mindset` (средний настрой на рост в школе), `school_achievement`, `school_ethnic_minority`, `school_poverty`, `school_size`. Категориальные признаки кодируем индикаторами. Ученики одной школы похожи, поэтому фолды cross-fitting разбиваем по школам (`groups`).

In [ ]:
URL = 'https://raw.githubusercontent.com/matheusfacure/python-causality-handbook/master/causal-inference-for-the-brave-and-true/data/learning_mindset.csv'
lm = pd.read_csv(URL)
Xm = pd.get_dummies(lm.drop(columns=['schoolid', 'intervention', 'achievement_score']),
                    columns=['ethnicity', 'gender', 'school_urbanicity'], drop_first=True).astype(float)
Tm, Ym = lm.intervention.values, lm.achievement_score.values
print('учеников:', len(lm), '| школ:', lm.schoolid.nunique(), '| доля получивших программу:', round(Tm.mean(), 3))
print('наивная разность средних исходов:', round(Ym[Tm == 1].mean() - Ym[Tm == 0].mean(), 3))

cfm = CausalForestDML(model_y=GradientBoostingRegressor(max_depth=3, random_state=SEED),
                      model_t=GradientBoostingClassifier(max_depth=3, random_state=SEED),
                      discrete_treatment=True, n_estimators=1000, min_samples_leaf=50, cv=5, random_state=SEED)
cfm.fit(Ym, Tm, X=Xm.values, groups=lm.schoolid.values)
print(cfm.ate__inference().summary_frame().round(3))
cate_m = cfm.effect(Xm.values)
print('квантили оценки CATE (5%, 50%, 95%):', np.round(np.quantile(cate_m, [0.05, 0.5, 0.95]), 3))

Важность признаков леса `feature_importances_` показывает, по каким признакам деревья чаще режут выборку, то есть где лес ищет различия эффекта. Это не важность для прогноза исхода: признак, сильно влияющий на успеваемость, но не на эффект программы, здесь будет внизу.

In [ ]:
fi = pd.Series(cfm.feature_importances_, index=Xm.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(fi.index[-10:], fi.values[-10:], color='#3b6ea5')
ax.set_xlabel('Causal forest feature importance (split frequency, depth-weighted)')
ax.set_title('Where the forest finds effect heterogeneity')
plt.tight_layout()
plt.show()

**Задание 5. Разбейте учеников на квартили по `school_mindset` (`pd.qcut(..., 4)`) и для каждого квартиля выведите среднюю оценку CATE и средний 95-процентный доверительный интервал оценки CATE (`cfm.effect_interval(Xm.values, alpha=0.05)`). Нарисуйте средние с интервалами. В каком квартиле настроя школы программа дает больший оцененный эффект?**

In [ ]:
# Ваш код здесь

Средний интервал по квартилю — это не интервал для среднего эффекта в квартиле: он описывает типичную неопределенность оценки CATE у одного ученика. Для вывода о подгруппе нужна групповая проверка — калибровка по квантилям и кривая Qini из тетради [«Гетерогенные эффекты»](https://colab.research.google.com/github/SadSabrina/open-xai-materials/blob/main/homeworks/PRACTICE_heterogeneous_effects.ipynb).

## Реализации

- `econml.grf.CausalForest` — сам лес на $(X, T, Y)$, без моделей исхода и склонности; годится для рандомизированных экспериментов.
- `econml.dml.CausalForestDML` — лес на остатках Робинсона, как в R-learner; нужен, когда воздействие назначено не случайно.
- R-пакет [grf](https://grf-labs.github.io/grf/) — `causal_forest()`, а также `quantile_forest()` и `instrumental_forest()` — другие леса того же семейства generalized random forests.

## Что унести из тетради

- **Адаптивная оценка находит эффект в шуме.** Там, где эффекта нет ни у кого, лучший лист дерева показывает эффект около единицы на тех же данных и около нуля на другой половине.
- **Честное расщепление — условие интервалов, а не украшение.** С ним покрытие близко к номинальному при умеренном размере листа; крупные листья дают смещение, и покрытие падает даже у честного леса.
- **Лес — это взвешенные соседи.** Веса $\alpha_i(x)$ сосредоточены на объектах, похожих на $x$ по признакам эффекта, и почти не зависят от признаков, которые влияют только на исход.
- **Позитивность лес не проверяет.** В области без одной из групп лес экстраполирует, ошибается в 4–5 раз сильнее и выдает интервал лишь в полтора раза шире, чем там, где сравнение есть.
- **Важность признаков леса — про неоднородность эффекта**, а не про прогноз исхода.

Источники:
1. [Wager & Athey, 2018. Estimation and inference of heterogeneous treatment effects using random forests](https://arxiv.org/abs/1510.04342)
2. [Athey, Tibshirani & Wager, 2019. Generalized random forests](https://arxiv.org/abs/1610.01271)
3. [Athey & Wager, 2019. Estimating treatment effects with causal forests: an application](https://arxiv.org/abs/1902.07409)
4. [EconML](https://github.com/py-why/EconML), [grf](https://grf-labs.github.io/grf/)